# 데이터 로드 및 압축 해제

In [2]:
import os
import zipfile
from google.colab import drive

drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/canopy/od_gps_preprocessed_dataset.zip'
extract_path = '/content/od_gps_work'

if os.path.exists(zip_path):
    os.makedirs(extract_path, exist_ok=True)
    print(f"'{zip_path}' 압축 해제 중...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print(f"압축 해제 완료: {extract_path}")

    print("\n압축 해제된 주요 폴더 목록:")
    for item in os.listdir(extract_path):
        print(f" - {item}")
else:
    print(f"지정된 경로에 파일이 존재하지 않습니다: {zip_path}")

Mounted at /content/drive
'/content/drive/MyDrive/canopy/od_gps_preprocessed_dataset.zip' 압축 해제 중...
압축 해제 완료: /content/od_gps_work

압축 해제된 주요 폴더 목록:
 - od_gps_work


# 데이터 준비 및 전처리

In [3]:
import os
import glob
import json
import random
import pandas as pd
import numpy as np

# 1. 경로 설정
extract_path = "/content/od_gps_work"
gps_dirs = glob.glob(os.path.join(extract_path, "**/gps"), recursive=True)
label_dirs = glob.glob(os.path.join(extract_path, "**/label"), recursive=True)

GPS_DIR = gps_dirs[0] if gps_dirs else os.path.join(extract_path, "gps")
LABEL_DIR = label_dirs[0] if label_dirs else os.path.join(extract_path, "label")

TARGET_MODES = [0, 1, 2, 3, 5]
MODE_NAMES = {0: "Walk", 1: "Bike", 2: "Car", 3: "Bus", 5: "Subway"}

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def extract_short_term_features(df):
    """
    150초 같은 장기 피처를 배제하고 5~30초 단위의 단기 패턴 중심 피처만 추출
    """
    lat = df['latitude'].values
    lon = df['longitude'].values
    ts = df['timestamp'].values.astype(np.float64)

    ts_sec = ts / 1000.0 if ts[0] > 1e11 else ts

    dt = np.clip(np.diff(ts_sec, prepend=ts_sec[0]), 0.1, None)
    dist = np.zeros(len(df))
    dist[1:] = haversine(lat[:-1], lon[:-1], lat[1:], lon[1:])

    speed = (dist / dt) * 3.6
    acceleration = np.diff(speed, prepend=speed[0]) / dt

    df['speed'] = speed
    df['acceleration'] = acceleration
    df['distance'] = dist

    speed_series = pd.Series(speed)
    accel_series = pd.Series(acceleration)

    # 단기 시계열 윈도우 (5초, 15초, 30초 포인트 기준)
    windows = [5, 15, 30]
    for w in windows:
        df[f'speed_mean_{w}'] = speed_series.rolling(window=w, min_periods=1).mean()
        df[f'speed_std_{w}'] = speed_series.rolling(window=w, min_periods=1).std().fillna(0)
        df[f'accel_mean_{w}'] = accel_series.rolling(window=w, min_periods=1).mean()

        # 정차 비율 (3 km/h 미만인 구간을 정차/도보 후보로 정의)
        stop_condition = (speed_series < 3.0)
        df[f'stoppage_ratio_{w}'] = stop_condition.rolling(window=w, min_periods=1).mean()

    # 단기 방위각 변화율 (회전 패턴)
    if 'bearing' in df.columns:
        bearing_series = df['bearing']
        df['bearing_change'] = bearing_series.diff().abs().fillna(0)
        df['bearing_change_15'] = df['bearing_change'].rolling(window=15, min_periods=1).mean()

    return df

# 2. 파일 로드 및 전처리 실행
gps_files = glob.glob(os.path.join(GPS_DIR, "*.csv"))
SAMPLE_LIMIT = 1500
random.seed(42)
sampled_files = random.sample(gps_files, min(SAMPLE_LIMIT, len(gps_files)))

dataset_list = []
matched_count = 0
excluded_label6_count = 0

for trip_idx, fpath in enumerate(sampled_files):
    try:
        filename = os.path.basename(fpath)
        prefix = filename.split("-Dataset")[0] if "-Dataset" in filename else filename.replace(".csv", "")

        possible_labels = glob.glob(os.path.join(LABEL_DIR, f"{prefix}*"))
        if not possible_labels:
            continue

        label_path = possible_labels[0]
        with open(label_path, "r", encoding="utf-8") as f:
            label_data = json.load(f)

        trspts = {item['tid']: item['value'] for item in label_data.get('trspt', [])}

        if 6 in trspts.values():
            excluded_label6_count += 1
            continue

        df = pd.read_csv(fpath)
        if len(df) < 30:
            continue

        stimes = {item['tid']: item['value'] for item in label_data.get('stime', [])}
        etimes = {item['tid']: item['value'] for item in label_data.get('etime', [])}

        modes = []
        ts_arr = df['timestamp'].values
        for t in ts_arr:
            matched_mode = -1
            for tid in stimes:
                if stimes[tid] <= t <= etimes.get(tid, stimes[tid]):
                    matched_mode = trspts.get(tid, -1)
                    break
            modes.append(matched_mode)

        df['mode'] = modes

        if not df['mode'].isin(TARGET_MODES).all():
            continue

        if len(df) < 20:
            continue

        # 단기 피처 추출 함수 적용 (150초 장기 피처 제외됨)
        df = extract_short_term_features(df)
        df['trip_id'] = f"trip_{trip_idx}"
        dataset_list.append(df)
        matched_count += 1
    except Exception:
        pass

if dataset_list:
    full_df = pd.concat(dataset_list, ignore_index=True)
    print(f"전처리 완료: 성공 트립 {matched_count:,}개 / 총 포인트 {len(full_df):,}개")
    print(full_df['mode'].value_counts().rename(index=MODE_NAMES))
else:
    print("로드된 데이터가 없습니다.")

전처리 완료: 성공 트립 1,387개 / 총 포인트 2,924,448개
mode
Car       1441431
Bus        804544
Walk       289968
Subway     208392
Bike       180113
Name: count, dtype: int64


# 학습/검증/테스트 데이터 분할 (Trip-level Split)

In [4]:
from sklearn.model_selection import train_test_split

if 'full_df' in locals() and not full_df.empty:
    # 1. 고유한 trip_id 리스트 추출
    unique_trips = full_df['trip_id'].unique()

    # 2. Train 세트와 임시 세트(Val + Test)로 분할 (70% / 30%)
    train_trips, temp_trips = train_test_split(
        unique_trips,
        test_size=0.3,
        random_state=42
    )

    # 3. 임시 세트를 Validation과 Test로 50%씩 분할 (각각 전체의 15% / 15%)
    val_trips, test_trips = train_test_split(
        temp_trips,
        test_size=0.5,
        random_state=42
    )

    # 4. trip_id를 기준으로 실제 데이터프레임 필터링
    train_df = full_df[full_df['trip_id'].isin(train_trips)].copy()
    val_df = full_df[full_df['trip_id'].isin(val_trips)].copy()
    test_df = full_df[full_df['trip_id'].isin(test_trips)].copy()

    # 5. 분할 결과 검증 출력
    print(f"✨ [데이터 누수 방지] Trip 기준 데이터 분할 완료!")
    print(f" - Train 세트     : 트립 {len(train_trips):,>4}개 | 포인트 {len(train_df):,}개")
    print(f" - Validation 세트: 트립 {len(val_trips):,>4}개 | 포인트 {len(val_df):,}개")
    print(f" - Test 세트      : 트립 {len(test_trips):,>4}개 | 포인트 {len(test_df):,}개")

    # 안정성 검증: 트립이 겹치는지 교집합 확인
    assert set(train_trips).isdisjoint(set(val_trips)), "⚠️ Train과 Val 사이에 트립 겹침 발생"
    assert set(train_trips).isdisjoint(set(test_trips)), "⚠️ Train과 Test 사이에 트립 겹침 발생"
    assert set(val_trips).isdisjoint(set(test_trips)), "⚠️ Val과 Test 사이에 트립 겹침 발생"
    print("🔒 검증 통과: Train / Val / Test 간의 트립 데이터 누수가 없습니다.")

else:
    print("⚠️ full_df 데이터가 없습니다. 전처리 코드를 먼저 실행해주세요.")

✨ [데이터 누수 방지] Trip 기준 데이터 분할 완료!
 - Train 세트     : 트립 ,970개 | 포인트 2,073,407개
 - Validation 세트: 트립 ,208개 | 포인트 442,334개
 - Test 세트      : 트립 ,209개 | 포인트 408,707개
🔒 검증 통과: Train / Val / Test 간의 트립 데이터 누수가 없습니다.


# 데이터셋 저장

In [5]:
# 전처리와 데이터 분할이 모두 끝난 후 실행
save_path = '/content/drive/MyDrive/canopy/preprocessed_data'
os.makedirs(save_path, exist_ok=True)

train_df.to_parquet(os.path.join(save_path, 'train_df.parquet'))
val_df.to_parquet(os.path.join(save_path, 'val_df.parquet'))
test_df.to_parquet(os.path.join(save_path, 'test_df.parquet'))

print("✨ 전처리된 데이터셋을 드라이브에 안전하게 저장했습니다!")

✨ 전처리된 데이터셋을 드라이브에 안전하게 저장했습니다!


# 데이터셋 로드

In [39]:
import pandas as pd

save_path = '/content/drive/MyDrive/canopy/preprocessed_data'
train_df = pd.read_parquet(os.path.join(save_path, 'train_df.parquet'))
val_df = pd.read_parquet(os.path.join(save_path, 'val_df.parquet'))
test_df = pd.read_parquet(os.path.join(save_path, 'test_df.parquet'))

print(f"✨ 데이터 로드 완료! Train: {len(train_df):,}개 | Val: {len(val_df):,}개 | Test: {len(test_df):,}개")

✨ 데이터 로드 완료! Train: 1,990,545개 | Val: 459,769개 | Test: 413,414개


# Model 1 - 고재현율(High-Recall) Change Point 분할 함수

In [40]:
import pandas as pd
import numpy as np

def high_recall_change_point_segmentation(trip_df, speed_th=2.0, accel_th=0.8):
    """
    Model 1: High-Recall Change Point Detection
    - 목적: 이동수단 변화 및 환승 구간의 변화점을 절대 놓치지 않고(High-Recall) 모두 포착
    - 방식: 속도, 가속도, 정차 여부 조건을 조금 넉넉하게(민감하게) 주어 구간을 세밀하게 쪼개고 경계점 수집
    """
    segmented_dfs = []
    all_trip_change_points = {}

    for trip_id, group in trip_df.groupby('trip_id'):
        g = group.copy().sort_values('timestamp').reset_index(drop=True)

        # 1. 민감한 조건: 속도가 낮거나, 가속도의 변화가 심하거나, 정차 중인 구간을 모두 후보로 포착
        # 환승 구간 전후에는 반드시 감속/정차/저속(도보) 과정이 포함되므로 조건을 넉넉하게 설정합니다.
        g['is_change_candidate'] = (g['speed'] < speed_th) | (g['acceleration'].abs() > accel_th)

        # 2. 상태가 바뀔 때마다 세그먼트 ID 부여 (잘게 쪼개어 변화점 포착 확률 극대화)
        g['segment_id'] = (g['is_change_candidate'] != g['is_change_candidate'].shift()).cumsum()
        g['global_segment_id'] = f"{trip_id}_" + g['segment_id'].astype(str)

        segmented_dfs.append(g)

        # 3. 세그먼트의 시작점과 끝점을 Change Point(변화점)로 전부 수집
        # 모델 예측값(또는 후처리)에 이 변화점들이 빠짐없이 포함되도록 보장합니다.
        change_points = []
        for _, seg_group in g.groupby('segment_id'):
            change_points.append(seg_group.index[0])
            change_points.append(seg_group.index[-1])

        all_trip_change_points[trip_id] = sorted(list(set(change_points)))

    result_df = pd.concat(segmented_dfs, ignore_index=True)
    return result_df, all_trip_change_points

# ==========================================
# 실제 데이터셋에 모델 1 적용 예시
# ==========================================
print("📌 [모델 1] 고재현율 변화점 탐지 및 세그먼트 분할 수행 중...")

train_df, train_cps = high_recall_change_point_segmentation(train_df)
val_df, val_cps = high_recall_change_point_segmentation(val_df)
test_df, test_cps = high_recall_change_point_segmentation(test_df)

print("✨ 모델 1 분할 완료! (환승 구간 변화점 확보 완료)")
print(f" - Train 트립별 평균 변화점 수: {np.mean([len(v) for v in train_cps.values()]):.1f}개")
print(f" - Test 트립별 평균 변화점 수 : {np.mean([len(v) for v in test_cps.values()]):.1f}개")

📌 [모델 1] 고재현율 변화점 탐지 및 세그먼트 분할 수행 중...
✨ 모델 1 분할 완료! (환승 구간 변화점 확보 완료)
 - Train 트립별 평균 변화점 수: 738.1개
 - Test 트립별 평균 변화점 수 : 696.3개


In [41]:
import numpy as np

def evaluate_change_point_with_lag_tolerance(df, trip_cps_dict, tolerance=30):
    """
    팀원분이 언급한 '20~30초의 전환 지연 시그니처'를 고려한 Change Point 성능 평가 함수
    - tolerance: 실제 전환점 기준으로 앞뒤 몇 초(포인트) 내에 예측 변화점이 있으면 정답으로 인정할 것인가? (기본값: 30초)
    """
    total_true_transitions = 0
    total_pred_changes = 0
    matched_true = 0
    matched_pred = 0

    # 1. 전체 테스트 데이터의 GPS 포인트 총 개수 측정
    total_gps_points = len(df)

    for trip_id, group in df.groupby('trip_id'):
        if trip_id not in trip_cps_dict:
            continue

        g = group.sort_values('timestamp').reset_index(drop=True)
        modes = g['mode'].values

        # 실제 모드가 바뀐 지점 (True Transition Indices) 찾기
        true_transitions = np.where(modes[1:] != modes[:-1])[0] + 1
        total_true_transitions += len(true_transitions)

        # 모델 1이 예측한 변화점 (Predicted Change Points)
        pred_cps = trip_cps_dict[trip_id]
        total_pred_changes += len(pred_cps)

        # Recall 검증: 실제 전환점 전후 [tolerance] 범위 내에 모델이 잡은 변화점이 있는가?
        for t_pt in true_transitions:
            if any(abs(t_pt - p_pt) <= tolerance for p_pt in pred_cps):
                matched_true += 1

        # Precision 검증: 모델이 예측한 변화점 전후 [tolerance] 범위 내에 실제 전환점이 있는가?
        for p_pt in pred_cps:
            if any(abs(p_pt - t_pt) <= tolerance for t_pt in true_transitions):
                matched_pred += 1

    # 성능 지표 계산
    recall = matched_true / total_true_transitions if total_true_transitions > 0 else 0
    precision = matched_pred / total_pred_changes if total_pred_changes > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\n================ [ 20~30초 지연 반영 환승 탐지 평가 결과 ] ================")
    print(f" - 테스트 데이터 총 GPS 포인트 수  : {total_gps_points:,}개")
    print(f" - 허용 오차 윈도우 (Tolerance)    : ±{tolerance}초 (포인트)")
    print(f" - 총 실제 환승 지점 수            : {total_true_transitions:,}개")
    print(f" - 모델 1이 탐지한 변화점 수       : {total_pred_changes:,}개")
    print(f" - 윈도우 내 매칭된 실제 환승 지점 : {matched_true:,}개")
    print(f"--------------------------------------------------------------------------")
    print(f" - Recall (재현율)    : {recall:.4f}  (환승 시그니처를 놓치지 않은 비율)")
    print(f" - Precision (정밀도) : {precision:.4f}  (잡은 것 중 실제 환승 구간과 인접한 비율)")
    print(f" - F1-Score           : {f1:.4f}")
    print(f"==========================================================================")

    return precision, recall, f1

# Test 셋에 대해 30초 지연 시그니처를 반영한 평가 실행
print("📌 [평가 진행 중] 20~30초 지연 시그니처를 반영하여 모델 1 성능 측정...")
eval_precision, eval_recall, eval_f1 = evaluate_change_point_with_lag_tolerance(test_df, test_cps, tolerance=30)

📌 [평가 진행 중] 20~30초 지연 시그니처를 반영하여 모델 1 성능 측정...

================ [ 20~30초 지연 반영 환승 탐지 평가 결과 ] ================
 - 테스트 데이터 총 GPS 포인트 수  : 413,414개
 - 허용 오차 윈도우 (Tolerance)    : ±30초 (포인트)
 - 총 실제 환승 지점 수            : 159개
 - 모델 1이 탐지한 변화점 수       : 144,836개
 - 윈도우 내 매칭된 실제 환승 지점 : 153개
--------------------------------------------------------------------------
 - Recall (재현율)    : 0.9623  (환승 시그니처를 놓치지 않은 비율)
 - Precision (정밀도) : 0.0213  (잡은 것 중 실제 환승 구간과 인접한 비율)
 - F1-Score           : 0.0417


In [36]:
import numpy as np

def evaluate_change_point_with_lag_tolerance(df, trip_cps_dict, tolerance=10):
    """
    팀원분이 언급한 '20~30초의 전환 지연 시그니처'를 고려한 Change Point 성능 평가 함수
    - tolerance: 실제 전환점 기준으로 앞뒤 몇 초(포인트) 내에 예측 변화점이 있으면 정답으로 인정할 것인가?
    """
    total_true_transitions = 0
    total_pred_changes = 0
    matched_true = 0
    matched_pred = 0

    # 1. 전체 테스트 데이터의 GPS 포인트 총 개수 측정
    total_gps_points = len(df)

    for trip_id, group in df.groupby('trip_id'):
        if trip_id not in trip_cps_dict:
            continue

        g = group.sort_values('timestamp').reset_index(drop=True)
        modes = g['mode'].values

        # 실제 모드가 바뀐 지점 (True Transition Indices) 찾기
        true_transitions = np.where(modes[1:] != modes[:-1])[0] + 1
        total_true_transitions += len(true_transitions)

        # 모델 1이 예측한 변화점 (Predicted Change Points)
        pred_cps = trip_cps_dict[trip_id]
        total_pred_changes += len(pred_cps)

        # Recall 검증: 실제 전환점 전후 [tolerance] 범위 내에 모델이 잡은 변화점이 있는가?
        for t_pt in true_transitions:
            if any(abs(t_pt - p_pt) <= tolerance for p_pt in pred_cps):
                matched_true += 1

        # Precision 검증: 모델이 예측한 변화점 전후 [tolerance] 범위 내에 실제 전환점이 있는가?
        for p_pt in pred_cps:
            if any(abs(p_pt - t_pt) <= tolerance for t_pt in true_transitions):
                matched_pred += 1

    # 성능 지표 계산
    recall = matched_true / total_true_transitions if total_true_transitions > 0 else 0
    precision = matched_pred / total_pred_changes if total_pred_changes > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\n================ [ 허용 오차 ±{tolerance}초 반영 환승 탐지 평가 결과 ] ================")
    print(f" - 테스트 데이터 총 GPS 포인트 수  : {total_gps_points:,}개")
    print(f" - 허용 오차 윈도우 (Tolerance)    : ±{tolerance}초 (포인트)")
    print(f" - 총 실제 환승 지점 수            : {total_true_transitions:,}개")
    print(f" - 모델 1이 탐지한 변화점 수       : {total_pred_changes:,}개")
    print(f" - 윈도우 내 매칭된 실제 환승 지점 : {matched_true:,}개")
    print(f"--------------------------------------------------------------------------")
    print(f" - Recall (재현율)    : {recall:.4f}  (환승 시그니처를 놓치지 않은 비율)")
    print(f" - Precision (정밀도) : {precision:.4f}  (잡은 것 중 실제 환승 구간과 인접한 비율)")
    print(f" - F1-Score           : {f1:.4f}")
    print(f"==========================================================================")

    return precision, recall, f1

# Test 셋에 대해 허용 오차를 10으로 낮춰서 실행
print("📌 [평가 진행 중] 허용 오차 ±10초로 변경하여 모델 1 성능 측정...")
eval_precision, eval_recall, eval_f1 = evaluate_change_point_with_lag_tolerance(test_df, test_cps, tolerance=10)

📌 [평가 진행 중] 허용 오차 ±10초로 변경하여 모델 1 성능 측정...

================ [ 허용 오차 ±10초 반영 환승 탐지 평가 결과 ] ================
 - 테스트 데이터 총 GPS 포인트 수  : 413,414개
 - 허용 오차 윈도우 (Tolerance)    : ±10초 (포인트)
 - 총 실제 환승 지점 수            : 159개
 - 모델 1이 탐지한 변화점 수       : 144,836개
 - 윈도우 내 매칭된 실제 환승 지점 : 137개
--------------------------------------------------------------------------
 - Recall (재현율)    : 0.8616  (환승 시그니처를 놓치지 않은 비율)
 - Precision (정밀도) : 0.0074  (잡은 것 중 실제 환승 구간과 인접한 비율)
 - F1-Score           : 0.0147


In [37]:
import numpy as np

def evaluate_change_point_with_lag_tolerance(df, trip_cps_dict, tolerance=20):
    """
    팀원분이 언급한 '20~30초의 전환 지연 시그니처'를 고려한 Change Point 성능 평가 함수
    - tolerance: 실제 전환점 기준으로 앞뒤 몇 초(포인트) 내에 예측 변화점이 있으면 정답으로 인정할 것인가?
    """
    total_true_transitions = 0
    total_pred_changes = 0
    matched_true = 0
    matched_pred = 0

    # 1. 전체 테스트 데이터의 GPS 포인트 총 개수 측정
    total_gps_points = len(df)

    for trip_id, group in df.groupby('trip_id'):
        if trip_id not in trip_cps_dict:
            continue

        g = group.sort_values('timestamp').reset_index(drop=True)
        modes = g['mode'].values

        # 실제 모드가 바뀐 지점 (True Transition Indices) 찾기
        true_transitions = np.where(modes[1:] != modes[:-1])[0] + 1
        total_true_transitions += len(true_transitions)

        # 모델 1이 예측한 변화점 (Predicted Change Points)
        pred_cps = trip_cps_dict[trip_id]
        total_pred_changes += len(pred_cps)

        # Recall 검증: 실제 전환점 전후 [tolerance] 범위 내에 모델이 잡은 변화점이 있는가?
        for t_pt in true_transitions:
            if any(abs(t_pt - p_pt) <= tolerance for p_pt in pred_cps):
                matched_true += 1

        # Precision 검증: 모델이 예측한 변화점 전후 [tolerance] 범위 내에 실제 전환점이 있는가?
        for p_pt in pred_cps:
            if any(abs(p_pt - t_pt) <= tolerance for t_pt in true_transitions):
                matched_pred += 1

    # 성능 지표 계산
    recall = matched_true / total_true_transitions if total_true_transitions > 0 else 0
    precision = matched_pred / total_pred_changes if total_pred_changes > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\n================ [ 허용 오차 ±{tolerance}초 반영 환승 탐지 평가 결과 ] ================")
    print(f" - 테스트 데이터 총 GPS 포인트 수  : {total_gps_points:,}개")
    print(f" - 허용 오차 윈도우 (Tolerance)    : ±{tolerance}초 (포인트)")
    print(f" - 총 실제 환승 지점 수            : {total_true_transitions:,}개")
    print(f" - 모델 1이 탐지한 변화점 수       : {total_pred_changes:,}개")
    print(f" - 윈도우 내 매칭된 실제 환승 지점 : {matched_true:,}개")
    print(f"--------------------------------------------------------------------------")
    print(f" - Recall (재현율)    : {recall:.4f}  (환승 시그니처를 놓치지 않은 비율)")
    print(f" - Precision (정밀도) : {precision:.4f}  (잡은 것 중 실제 환승 구간과 인접한 비율)")
    print(f" - F1-Score           : {f1:.4f}")
    print(f"==========================================================================")

    return precision, recall, f1

# Test 셋에 대해 허용 오차를 20으로 설정하여 실행
print("📌 [평가 진행 중] 허용 오차 ±20초로 변경하여 모델 1 성능 측정...")
eval_precision, eval_recall, eval_f1 = evaluate_change_point_with_lag_tolerance(test_df, test_cps, tolerance=20)

📌 [평가 진행 중] 허용 오차 ±20초로 변경하여 모델 1 성능 측정...

================ [ 허용 오차 ±20초 반영 환승 탐지 평가 결과 ] ================
 - 테스트 데이터 총 GPS 포인트 수  : 413,414개
 - 허용 오차 윈도우 (Tolerance)    : ±20초 (포인트)
 - 총 실제 환승 지점 수            : 159개
 - 모델 1이 탐지한 변화점 수       : 144,836개
 - 윈도우 내 매칭된 실제 환승 지점 : 145개
--------------------------------------------------------------------------
 - Recall (재현율)    : 0.9119  (환승 시그니처를 놓치지 않은 비율)
 - Precision (정밀도) : 0.0141  (잡은 것 중 실제 환승 구간과 인접한 비율)
 - F1-Score           : 0.0278


In [33]:
import random
import numpy as np

def sample_trip_transition_visualization(df, trip_cps_dict, tolerance=30, num_samples=5):
    """
    실제 환승(전환)이 발생한 트립들을 샘플링하여
    실제 환승 지점과 모델 1의 예측 변화점 간의 매칭 상태를 확인하는 함수
    """
    valid_trips = []

    # 1. 실제 환승 지점이 최소 1개 이상 존재하는 트립들만 수집
    for trip_id, group in df.groupby('trip_id'):
        if trip_id not in trip_cps_dict:
            continue
        g = group.sort_values('timestamp').reset_index(drop=True)
        modes = g['mode'].values

        true_transitions = np.where(modes[1:] != modes[:-1])[0] + 1
        if len(true_transitions) > 0:
            valid_trips.append((trip_id, g, true_transitions, trip_cps_dict[trip_id]))

    # 2. 무작위로 num_samples(5개) 선정
    if len(valid_trips) == 0:
        print("⚠️ 환승 지점이 포함된 트립이 없습니다.")
        return

    sampled = random.sample(valid_trips, min(num_samples, len(valid_trips)))

    print(f"✨ [샘플 분석 결과] 총 {len(sampled)}개 트립 대조 (허용 오차 윈도우: ±{tolerance}포인트)\n")

    for idx, (trip_id, g, true_trans, pred_cps) in enumerate(sampled):
        print(f"--------------------------------------------------")
        print(f"📌 [샘플 {idx+1}] Trip ID: {trip_id} (총 길이: {len(g):,} 포인트)")
        print(f"--------------------------------------------------")
        print(f"  [실제 환승 지점 및 모드 변화]")

        for t_pt in true_trans:
            prev_mode = g.iloc[t_pt - 1]['mode']
            curr_mode = g.iloc[t_pt]['mode']

            # 30초(tolerance) 내에 모델이 예측한 변화점이 있는지 확인
            matched_pnts = [p for p in pred_cps if abs(t_pt - p) <= tolerance]
            is_matched = len(matched_pnts) > 0

            status = "✅ [포착 성공]" if is_matched else "❌ [놓침]"
            print(f"   • Index {t_pt:4d} | {prev_mode} ➔ {curr_mode}  ==>  {status}")
            if is_matched:
                print(f"     (가장 가까운 모델 예측 변화점: 인덱스 {matched_pnts[0]})")

        print(f"  • 모델 1이 이 트립에서 탐지한 총 변화점 수: {len(pred_cps):,}개\n")

# 실행
sample_trip_transition_visualization(test_df, test_cps, tolerance=30, num_samples=5)

✨ [샘플 분석 결과] 총 5개 트립 대조 (허용 오차 윈도우: ±30포인트)

--------------------------------------------------
📌 [샘플 1] Trip ID: trip_89 (총 길이: 198 포인트)
--------------------------------------------------
  [실제 환승 지점 및 모드 변화]
   • Index   33 | 5 ➔ 0  ==>  ✅ [포착 성공]
     (가장 가까운 모델 예측 변화점: 인덱스 10)
  • 모델 1이 이 트립에서 탐지한 총 변화점 수: 109개

--------------------------------------------------
📌 [샘플 2] Trip ID: trip_884 (총 길이: 6,765 포인트)
--------------------------------------------------
  [실제 환승 지점 및 모드 변화]
   • Index 2287 | 0 ➔ 2  ==>  ✅ [포착 성공]
     (가장 가까운 모델 예측 변화점: 인덱스 2275)
  • 모델 1이 이 트립에서 탐지한 총 변화점 수: 1,967개

--------------------------------------------------
📌 [샘플 3] Trip ID: trip_722 (총 길이: 1,279 포인트)
--------------------------------------------------
  [실제 환승 지점 및 모드 변화]
   • Index  517 | 1 ➔ 5  ==>  ✅ [포착 성공]
     (가장 가까운 모델 예측 변화점: 인덱스 489)
   • Index  673 | 5 ➔ 0  ==>  ✅ [포착 성공]
     (가장 가까운 모델 예측 변화점: 인덱스 643)
  • 모델 1이 이 트립에서 탐지한 총 변화점 수: 662개

--------------------------------------------------
📌

## ±30초(30포인트) 허용 오차 윈도우 안에 들지 못해 False Negative로 분류된 실제 환승 포인트

In [26]:
import numpy as np
import pandas as pd

def inspect_closest_change_points(df, trip_cps_dict, tolerance=30):
    """
    놓친 환승 포인트들에 대해 모델 1이 예측한 가장 가까운 변화점의 위치, 거리,
    그리고 인근(±100이내) 예측점 목록을 '예측점 목록(n개): [...]' 형식으로 출력하는 함수
    """
    print(f"=== [놓친 환승 포인트 심층 분석: 가장 가까운 예측 변화점 및 상태 확인] ===\n")
    missed_count = 0

    for trip_id, group in df.groupby('trip_id'):
        if trip_id not in trip_cps_dict:
            continue

        g = group.sort_values('timestamp').reset_index(drop=True)
        modes = g['mode'].values

        # 실제 모드가 바뀐 지점 찾기
        true_transitions = np.where(modes[1:] != modes[:-1])[0] + 1
        pred_cps = trip_cps_dict[trip_id]

        for t_pt in true_transitions:
            # 30초 윈도우 내에 예측점이 없는 경우만 필터링 (놓친 포인트)
            is_matched = any(abs(t_pt - p_pt) <= tolerance for p_pt in pred_cps)

            if not is_matched:
                missed_count += 1
                prev_mode = modes[t_pt - 1]
                curr_mode = modes[t_pt]

                # 1. 모델이 예측한 전체 변화점 중 가장 가까운 것 찾기
                if pred_cps:
                    closest_cp = min(pred_cps, key=lambda x: abs(x - t_pt))
                    distance = abs(closest_cp - t_pt)
                else:
                    closest_cp = None
                    distance = -1

                # 2. 실제 전환점 기준 전후 ±100포인트 내에 있는 모델 예측 변화점들 및 개수 추출
                nearby_cps = [p for p in pred_cps if abs(p - t_pt) <= 100]
                nearby_count = len(nearby_cps)

                # 3. 해당 시점 주변(전후 5포인트) 평균 속도 및 가속도 계산
                start_idx = max(0, t_pt - 5)
                end_idx = min(len(g) - 1, t_pt + 5)
                surrounding_df = g.loc[start_idx:end_idx]

                avg_speed = surrounding_df['speed'].mean()
                avg_accel = surrounding_df['acceleration'].abs().mean()
                point_data = g.loc[t_pt]

                # 4. 결과 출력 포맷 반영
                print(f"❌ [놓친 환승 #{missed_count}]")
                print(f"   • Trip ID          : {trip_id}")
                print(f"   • 실제 전환 인덱스 : {t_pt}  ({prev_mode} ➔ {curr_mode})")
                print(f"   • 가장 가까운 예측점 : 인덱스 {closest_cp} (실제와 **{distance}포인트** 떨어짐)")
                print(f"   • 인근(±100이내) 예측점 목록({nearby_count}개): {nearby_cps}")
                print(f"   • 주변 평균속도    : {avg_speed:.2f} m/s (약 {avg_speed * 3.6:.1f} km/h)")
                print(f"   • 주변 평균가속    : {avg_accel:.2f} m/s²")
                print(f"   • 해당 시점 상태   : 속도={point_data.get('speed', 0):.4f} m/s, 가속도={point_data.get('acceleration', 0):.4f} m/s²")
                print("-" * 65)

# 테스트 셋 기준으로 실행
inspect_closest_change_points(test_df, test_cps, tolerance=30)

=== [놓친 환승 포인트 심층 분석: 가장 가까운 예측 변화점 및 상태 확인] ===

❌ [놓친 환승 #1]
   • Trip ID          : trip_1087
   • 실제 전환 인덱스 : 265  (0 ➔ 3)
   • 가장 가까운 예측점 : 인덱스 422 (실제와 **157포인트** 떨어짐)
   • 인근(±100이내) 예측점 목록(0개): []
   • 주변 평균속도    : 0.09 m/s (약 0.3 km/h)
   • 주변 평균가속    : 0.09 m/s²
   • 해당 시점 상태   : 속도=0.0861 m/s, 가속도=-0.2766 m/s²
-----------------------------------------------------------------
❌ [놓친 환승 #2]
   • Trip ID          : trip_1093
   • 실제 전환 인덱스 : 1287  (3 ➔ 5)
   • 가장 가까운 예측점 : 인덱스 1325 (실제와 **38포인트** 떨어짐)
   • 인근(±100이내) 예측점 목록(28개): [np.int64(1325), np.int64(1326), np.int64(1327), np.int64(1329), np.int64(1330), np.int64(1331), np.int64(1334), np.int64(1335), np.int64(1336), np.int64(1337), np.int64(1338), np.int64(1339), np.int64(1340), np.int64(1341), np.int64(1357), np.int64(1358), np.int64(1359), np.int64(1366), np.int64(1367), np.int64(1368), np.int64(1369), np.int64(1370), np.int64(1371), np.int64(1372), np.int64(1373), np.int64(1382), np.int64(1383), np.int64(1384)]
   • 주변 

# 모델 2 - LightGBM 다중 클래스 분류기

In [45]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score

# 0. Haversine 거리 계산 함수 (피처 추출용)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # 지구 반지름 (m)
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def calculate_bearing(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dLon = lon2 - lon1
    x = np.sin(dLon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dLon)
    return np.degrees(np.arctan2(x, y))

# 1. 단기 + 장기 복합 피처 추출 함수 (v3)
def extract_advanced_features_v3(df):
    processed_dfs = []
    for trip_id, group in df.groupby('trip_id'):
        group = group.reset_index(drop=True)
        lat = group['latitude'].values
        lon = group['longitude'].values
        ts = group['timestamp'].values.astype(np.float64)

        ts_sec = ts / 1000.0 if ts[0] > 1e11 else ts
        dt = np.clip(np.diff(ts_sec, prepend=ts_sec[0]), 0.1, None)

        dist = np.zeros(len(group))
        dist[1:] = haversine(lat[:-1], lon[:-1], lat[1:], lon[1:])

        speed = (dist / dt) * 3.6
        acceleration = np.diff(speed, prepend=speed[0]) / dt

        group['speed'] = speed
        group['acceleration'] = acceleration
        group['distance'] = dist

        bearings = np.zeros(len(group))
        if len(group) > 1:
            bearings[1:] = calculate_bearing(lat[:-1], lon[:-1], lat[1:], lon[1:])
        bearing_diff = np.abs(np.diff(bearings, prepend=bearings[0]))
        bearing_diff = np.where(bearing_diff > 180, 360 - bearing_diff, bearing_diff)
        group['bearing_change'] = bearing_diff

        speed_series = pd.Series(speed)
        accel_series = pd.Series(acceleration)

        # 단기 윈도우 (변화 및 반응 감지)
        group['speed_mean_5'] = speed_series.rolling(window=5, min_periods=1).mean()
        group['speed_std_5'] = speed_series.rolling(window=5, min_periods=1).std().fillna(0)
        group['speed_max_10'] = speed_series.rolling(window=10, min_periods=1).max()

        # 중장기 윈도우 (패턴 및 문맥 파악: 정차 비율, 분위수, 장기 통계)
        group['speed_mean_30'] = speed_series.rolling(window=30, min_periods=1).mean()
        group['speed_std_30'] = speed_series.rolling(window=30, min_periods=1).std().fillna(0)
        group['speed_max_60'] = speed_series.rolling(window=60, min_periods=1).max()
        group['speed_mean_150'] = speed_series.rolling(window=150, min_periods=1).mean()

        is_stopped = (speed < 3.0).astype(int)
        stopped_series = pd.Series(is_stopped)
        group['stop_count_60'] = stopped_series.rolling(window=60, min_periods=1).sum()
        group['stop_count_150'] = stopped_series.rolling(window=150, min_periods=1).sum()
        group['stoppage_ratio_60'] = stopped_series.rolling(window=60, min_periods=1).mean()

        group['speed_q25_60'] = speed_series.rolling(window=60, min_periods=1).quantile(0.25)
        group['speed_q75_60'] = speed_series.rolling(window=60, min_periods=1).quantile(0.75)
        group['accel_std_30'] = accel_series.rolling(window=30, min_periods=1).std().fillna(0)

        processed_dfs.append(group)

    return pd.concat(processed_dfs, ignore_index=True)

print("📌 [전처리] Train, Validation, Test 세트에 단기+장기 피처 추출 중...")
train_df = extract_advanced_features_v3(train_df)
val_df = extract_advanced_features_v3(val_df)
test_df = extract_advanced_features_v3(test_df)

# 2. 사용할 피처 목록 정의 (단기 + 장기 조합)
features = [
    'speed', 'acceleration', 'distance', 'bearing_change',
    'speed_mean_5', 'speed_std_5', 'speed_max_10',
    'speed_mean_30', 'speed_std_30', 'speed_max_60', 'speed_mean_150',
    'stop_count_60', 'stop_count_150', 'stoppage_ratio_60',
    'speed_q25_60', 'speed_q75_60', 'accel_std_30'
]
target_col = 'mode'

# 3. 라벨 매핑 (0, 1, 2, 3, 5 -> 0, 1, 2, 3, 4)
label_map = {0: 0, 1: 1, 2: 2, 3: 3, 5: 4}
inv_label_map = {0: 0, 1: 1, 2: 2, 3: 3, 4: 5}

for df in [train_df, val_df, test_df]:
    df['mapped_mode'] = df[target_col].map(label_map)

X_train, y_train = train_df[features], train_df['mapped_mode']
X_val, y_val = val_df[features], val_df['mapped_mode']
X_test, y_test = test_df[features], test_df['mapped_mode']

print("🚀 [모델 2] LightGBM 학습 시작 (단기+장기 피처 반영)...")

# 4. LightGBM 모델 정의 및 학습
model2 = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=5,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

model2.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=True), lgb.log_evaluation(50)]
)

# 5. 예측 및 원본 라벨 복원 평가
print("\n📊 [모델 2] Test 세트 예측 및 성능 평가 중...")
y_pred_mapped = model2.predict(X_test)

y_test_original = y_test.map(inv_label_map)
y_pred_original = pd.Series(y_pred_mapped).map(inv_label_map)

target_names = ['Walk (0)', 'Bike (1)', 'Car (2)', 'Bus (3)', 'Subway (5)']
print("\n" + "="*50)
print(" 🎯 LightGBM (모델 2 - 단기+장기 피처 조합) 최종 리포트 ")
print("="*50)
print(classification_report(y_test_original, y_pred_original, target_names=target_names))
print(f"🔹 최종 정확도 (Accuracy): {accuracy_score(y_test_original, y_pred_original):.4f}")
print("="*50)

📌 [전처리] Train, Validation, Test 세트에 단기+장기 피처 추출 중...
🚀 [모델 2] LightGBM 학습 시작 (단기+장기 피처 반영)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.090480 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4037
[LightGBM] [Info] Number of data points in the train set: 1990545, number of used features: 17
[LightGBM] [Info] Start training from score -2.306000
[LightGBM] [Info] Start training from score -3.002803
[LightGBM] [Info] Start training from score -0.698390
[LightGBM] [Info] Start training from score -1.319797
[LightGBM] [Info] Start training from score -2.452035
Training until validation scores don't improve for 30 rounds
[50]	valid_0's multi_logloss: 0.905748
[100]	valid_0's multi_logloss: 0.872134
[150]	valid_0's multi_logloss: 0.862753
[200]	valid_0's multi_logloss: 0.857776
[250]	valid_0's multi_logloss: 0.855075
[300]	valid_0's multi

# Post-processing: ① 최빈값 스무딩 필터, ② 최소 세그먼트 유지 제약

In [49]:
from scipy.stats import mode

# 6. 후처리 함수 정의 (스무딩 및 노이즈 제거)
def apply_post_processing(df, pred_col='pred_mapped', window_size=11, min_segment_len=10):
    """
    1. 최빈값(Mode) 기반 롤링 스무딩으로 단기 노이즈(Flickering) 제거
    2. 너무 짧은 세그먼트(min_segment_len 미만)를 앞뒤 맥락에 맞춰 병합
    """
    processed_dfs = []

    for trip_id, group in df.groupby('trip_id'):
        group = group.copy()
        preds = group[pred_col].values

        # --- [Step 1] Rolling Mode (최빈값) 스무딩 적용 ---
        # 홀수 크기의 윈도우를 사용해 주변에서 가장 많이 등장한 이동수단으로 덮어씀
        smoothed_preds = np.copy(preds)
        half_w = window_size // 2

        for i in range(len(preds)):
            start_idx = max(0, i - half_w)
            end_idx = min(len(preds), i + half_w + 1)
            window_vals = preds[start_idx:end_idx]

            # 최빈값 추출 (scipy mode 사용)
            m_val, count = mode(window_vals, keepdims=True)
            smoothed_preds[i] = m_val[0]

        # --- [Step 2] 최소 유지 시간(세그먼트 길이) 제약 적용 ---
        # 튀어나온 짧은 구간(예: 10초 미만의 환승 노이즈)을 인접 세그먼트로 병합
        cleaned_preds = np.copy(smoothed_preds)
        n = len(cleaned_preds)
        if n > 0:
            segment_start = 0
            for i in range(1, n + 1):
                # 세그먼트가 끝나거나 마지막 포인트일 때
                if i == n or cleaned_preds[i] != cleaned_preds[segment_start]:
                    segment_len = i - segment_start

                    # 만약 세그먼트 길이가 너무 짧고(노이즈로 판단), 양옆에 병합할 대상이 있다면
                    if segment_len < min_segment_len:
                        if segment_start > 0 and i < n:
                            # 양옆 중 앞쪽 세그먼트의 수단으로 흡수
                            cleaned_preds[segment_start:i] = cleaned_preds[segment_start - 1]
                        elif segment_start > 0:
                            cleaned_preds[segment_start:i] = cleaned_preds[segment_start - 1]
                        elif i < n:
                            cleaned_preds[segment_start:i] = cleaned_preds[i]

                    segment_start = i

        group['smoothed_mode'] = cleaned_preds
        processed_dfs.append(group)

    return pd.concat(processed_dfs, ignore_index=True)

print("\n🧹 [후처리] 모델 2 예측 결과에 스무딩 및 노이즈 제거 적용 중...")

# Test 데이터에 모델 예측 수행
test_df['pred_mapped'] = model2.predict(X_test)

# 후처리 실행 (window_size=11: 약 11초 윈도우 내 최빈값, min_segment_len=10: 10포인트 미만 짧은 튐 현상 제거)
test_df_processed = apply_post_processing(test_df, pred_col='pred_mapped', window_size=11, min_segment_len=10)

# 7. 후처리 후 성능 평가
y_test_original = test_df['mapped_mode'].map(inv_label_map)
y_pred_smoothed = test_df_processed['smoothed_mode'].map(inv_label_map)

target_names = ['Walk (0)', 'Bike (1)', 'Car (2)', 'Bus (3)', 'Subway (5)']
print("\n" + "="*50)
print(" 🎯 후처리(Smoothing) 적용 후 최종 성능 리포트 ")
print("="*50)
print(classification_report(y_test_original, y_pred_smoothed, target_names=target_names))
print(f"🔹 후처리 후 최종 정확도 (Accuracy): {accuracy_score(y_test_original, y_pred_smoothed):.4f}")
print("="*50)


🧹 [후처리] 모델 2 예측 결과에 스무딩 및 노이즈 제거 적용 중...

 🎯 후처리(Smoothing) 적용 후 최종 성능 리포트 
              precision    recall  f1-score   support

    Walk (0)       0.67      0.76      0.72     49123
    Bike (1)       0.62      0.32      0.42     12984
     Car (2)       0.72      0.84      0.78    220206
     Bus (3)       0.53      0.41      0.46     95428
  Subway (5)       0.73      0.47      0.58     35673

    accuracy                           0.68    413414
   macro avg       0.65      0.56      0.59    413414
weighted avg       0.67      0.68      0.67    413414

🔹 후처리 후 최종 정확도 (Accuracy): 0.6808


# 세그먼트 및 전환점(Transition) 탐지 결과

In [50]:
# 8. 전환점(Transition Point) 추출 및 ±30초 허용 오차 평가 함수
def evaluate_transitions(df, true_col='mode', pred_col='smoothed_mode', tolerance_sec=30):
    """
    실제 전환점과 예측된 전환점을 비교하여 ±30초 오차 내 매칭 여부(Precision, Recall) 평가
    """
    total_true_transitions = 0
    total_pred_transitions = 0
    matched_true_transitions = 0

    for trip_id, group in df.groupby('trip_id'):
        group = group.reset_index(drop=True)
        true_modes = group[true_col].values
        pred_modes = group[pred_col].values
        timestamps = group['timestamp'].values.astype(np.float64)
        ts_sec = timestamps / 1000.0 if timestamps[0] > 1e11 else timestamps

        # 실제 전환점 인덱스 (값이 바뀌는 지점)
        true_trans_idx = np.where(true_modes[:-1] != true_modes[1:])[0] + 1
        # 예측 전환점 인덱스
        pred_trans_idx = np.where(pred_modes[:-1] != pred_modes[1:])[0] + 1

        total_true_transitions += len(true_trans_idx)
        total_pred_transitions += len(pred_trans_idx)

        # ±30초 윈도우 매칭 평가
        matched = set()
        for t_idx in true_trans_idx:
            t_time = ts_sec[t_idx]
            # 허용 오차 범위 내에 예측 전환점이 있는지 확인
            for p_idx in pred_trans_idx:
                if p_idx not in matched:
                    p_time = ts_sec[p_idx]
                    if abs(t_time - p_time) <= tolerance_sec:
                        matched_true_transitions += 1
                        matched.add(p_idx)
                        break

    recall = matched_true_transitions / total_true_transitions if total_true_transitions > 0 else 0
    precision = matched_true_transitions / total_pred_transitions if total_pred_transitions > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return total_true_transitions, total_pred_transitions, matched_true_transitions, precision, recall, f1

# 전환점 평가 실행
true_trans_cnt, pred_trans_cnt, matched_cnt, p_trans, r_trans, f1_trans = evaluate_transitions(test_df_processed, true_col='mapped_mode', pred_col='smoothed_mode', tolerance_sec=30)

print("\n" + "="*50)
print(" 🔄 세그먼트 및 전환점(Transition) 탐지 결과 요약 ")
print("="*50)
print(f"🔹 총 GPS 포인트 개수: {len(test_df_processed):,} 개")
print(f"🔹 실제 전환 구간(True Transitions): {true_trans_cnt} 개")
print(f"🔹 모델이 포착한 전환 후보(Pred Transitions): {pred_trans_cnt:,} 개 (후처리로 대폭 정제됨)")
print(f"🔹 ±30초 오차 내 성공적으로 맞춘 전환점: {matched_cnt} 개")
print(f"   - 전환점 탐지 재현율 (Recall): {r_trans:.4f}")
print(f"   - 전환점 탐지 정밀도 (Precision): {p_trans:.4f}")
print(f"   - 전환점 탐지 F1-Score: {f1_trans:.4f}")
print("="*50)


# 9. 세그먼트 병합 결과 샘플 5개 출력 함수
def print_segment_samples(df, num_samples=5):
    """
    Test 데이터 중 무작위로 5개의 Trip을 골라 실제 세그먼트와 예측된 세그먼트를 비교 출력
    """
    unique_trips = df['trip_id'].unique()
    sample_trips = np.random.choice(unique_trips, size=min(num_samples, len(unique_trips)), replace=False)

    print(f"\n" + "="*70)
    print(f" 📋 [예시 샘플 {len(sample_trips)}개] 실제 세그먼트 vs 후처리 예측 세그먼트 비교 ")
    print("="*70)

    for idx, t_id in enumerate(sample_trips, 1):
        sub_df = df[df['trip_id'] == t_id].reset_index(drop=True)

        # 실제 세그먼트 압축 (연속된 동일 모드를 하나의 세그먼트로 묶음)
        true_modes = sub_df['mapped_mode'].map(inv_label_map).values
        true_segments = []
        if len(true_modes) > 0:
            curr_mode = true_modes[0]
            curr_len = 1
            for m in true_modes[1:]:
                if m == curr_mode:
                    curr_len += 1
                else:
                    true_segments.append(f"{curr_mode}({curr_len}s)")
                    curr_mode = m
                    curr_len = 1
            true_segments.append(f"{curr_mode}({curr_len}s)")

        # 예측 세그먼트 압축
        pred_modes = sub_df['smoothed_mode'].map(inv_label_map).values
        pred_segments = []
        if len(pred_modes) > 0:
            curr_mode = pred_modes[0]
            curr_len = 1
            for m in pred_modes[1:]:
                if m == curr_mode:
                    curr_len += 1
                else:
                    pred_segments.append(f"{curr_mode}({curr_len}s)")
                    curr_mode = m
                    curr_len = 1
            pred_segments.append(f"{curr_mode}({curr_len}s)")

        print(f"\n[Sample {idx}] Trip ID: {t_id} (총 포인트: {len(sub_df)}개)")
        print(f"  • [실제 경로]  : {' ➔ '.join(true_segments)}")
        print(f"  • [예측 경로]  : {' ➔ '.join(pred_segments)}")
    print("="*70)

# 샘플 출력 실행
print_segment_samples(test_df_processed, num_samples=5)


 🔄 세그먼트 및 전환점(Transition) 탐지 결과 요약 
🔹 총 GPS 포인트 개수: 413,414 개
🔹 실제 전환 구간(True Transitions): 159 개
🔹 모델이 포착한 전환 후보(Pred Transitions): 3,555 개 (후처리로 대폭 정제됨)
🔹 ±30초 오차 내 성공적으로 맞춘 전환점: 108 개
   - 전환점 탐지 재현율 (Recall): 0.6792
   - 전환점 탐지 정밀도 (Precision): 0.0304
   - 전환점 탐지 F1-Score: 0.0582

 📋 [예시 샘플 5개] 실제 세그먼트 vs 후처리 예측 세그먼트 비교 

[Sample 1] Trip ID: trip_171 (총 포인트: 2056개)
  • [실제 경로]  : 2(2056s)
  • [예측 경로]  : 2(121s) ➔ 3(18s) ➔ 2(59s) ➔ 3(17s) ➔ 2(277s) ➔ 3(88s) ➔ 2(11s) ➔ 3(13s) ➔ 2(157s) ➔ 3(168s) ➔ 0(59s) ➔ 3(143s) ➔ 2(18s) ➔ 3(195s) ➔ 2(492s) ➔ 3(14s) ➔ 2(140s) ➔ 3(66s)

[Sample 2] Trip ID: trip_865 (총 포인트: 2858개)
  • [실제 경로]  : 2(2858s)
  • [예측 경로]  : 2(86s) ➔ 1(11s) ➔ 2(46s) ➔ 3(443s) ➔ 2(158s) ➔ 5(57s) ➔ 2(56s) ➔ 5(14s) ➔ 2(309s) ➔ 5(62s) ➔ 2(25s) ➔ 3(60s) ➔ 2(109s) ➔ 1(24s) ➔ 3(74s) ➔ 0(142s) ➔ 2(439s) ➔ 0(53s) ➔ 2(690s)

[Sample 3] Trip ID: trip_636 (총 포인트: 3505개)
  • [실제 경로]  : 2(2703s) ➔ 0(331s) ➔ 1(471s)
  • [예측 경로]  : 1(53s) ➔ 2(153s) ➔ 5(15s) ➔ 2(34s) ➔ 5(15s) ➔ 2(25s) ➔ 5